##  LECTURE2 : Deep Neural Network

- 1교시: 신경망 정의하고 구조 살펴보기
- 2교시: Loss
- 3교시: Optimizer
- 4교시: Weight Initialize
- 5교시: Multi Layer Perceptron for MNIST
---

날짜 : 2023. 09. 07


## [1교시] 신경망 정의하고 구조 살펴보기

1. 데이터를 가져오기 위해 필요한 라이브러리들 불러오기
2. 신경망을 정의하고 초기화하기
3. 데이터를 모델에 적용해 테스트하기



###1. 데이터를 가져오기 위해 필요한 라이브러리들 불러오기

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Q. 대표적인 nn 모듈들, F 모듈들 따로 정리

- https://pytorch.org/docs/stable/nn.html

- https://pytorch.org/docs/stable/nn.functional.html



Q. torch.nn.functional vs torch.nn

겹치는 기능이 있는건 어떻게 하나요?


- torch.nn.functional: 기본적으로 함수
- torch.nn: 기본적으로 클래스

동작 자체에는 차이가 없다.
ref: https://discuss.pytorch.org/t/what-is-the-difference-between-torch-nn-and-torch-nn-functional/33597


다만 후자는 클래스이므로, 인스턴스화를 해야 사용할 수 있다는 점.

개발에 입맛에 맞는 것을 골라 쓰면된다.

아래 예제를 살펴보자

In [3]:
import torch
import torch.nn as nn

loss = nn.CrossEntropyLoss()        #인스턴스!
input = torch.randn(3, 5, requires_grad=True)
target = torch.empty(3, dtype=torch.long).random_(5)
output = loss(input, target)
output.backward()

In [4]:
import torch
import torch.nn.functional as F

input = torch.randn(3, 5, requires_grad=True)
target = torch.randint(5, (3,), dtype=torch.int64)
loss = F.cross_entropy(input, target)   #바로 함수로 적용
loss.backward()

실제 nn.conv1d의 코드 부분을 살펴보자
https://pytorch.org/docs/stable/_modules/torch/nn/modules/conv.html#Conv1d


```
class _ConvNd(Module):
    ...
    def _conv_forward(self, input: Tensor, weight: Tensor, bias: Optional[Tensor]):
        if self.padding_mode != 'zeros':
            return F.conv1d(F.pad(input, self._reversed_padding_repeated_twice, mode=self.padding_mode),
                            weight, bias, self.stride,
                            _single(0), self.dilation, self.groups)
        return F.conv1d(input, weight, bias, self.stride,
                        self.padding, self.dilation, self.groups)

    def forward(self, input: Tensor) -> Tensor:
        return self._conv_forward(input, self.weight, self.bias)

```

_conv_forward 메소드내에 F.conv1d를 호출하여 쓰는 것을 볼 수 있다.




#### 그러면 차이가 없는데 왜 둘다 있는건가요?


- torch.nn.CrossEntropyLoss(weight=None, size_average=None, ignore_index=- 100, reduce=None, reduction='mean', label_smoothing=0.0)

- torch.nn.functional.cross_entropy(input, target, weight=None, size_average=None, ignore_index=- 100, reduce=None, reduction='mean', label_smoothing=0.0)

weight (Tensor, optional) – a manual rescaling weight given to each class. If given, has to be a Tensor of size C

In [6]:
# define the weights for each class
weights = torch.tensor([1.0, 2.0, 1.0])

"""
class CrossEntropyLoss:
  def __init__(self, weight):
    self.weight = weight
    #여기 self에 weight를 저장해두고 두고두고 사용할 수 있게 된다는 점!
"""

# define the CrossEntropyLoss with weights
criterion = nn.CrossEntropyLoss(weight=weights)

# define the inputs and labels
inputs = torch.randn(10, 3)
labels = torch.tensor([0, 1, 2, 1, 0, 2, 0, 1, 2, 0]) # (10)

# calculate the loss
loss = criterion(inputs, labels)
print(f"nn = {loss}")

import torch.nn.functional as F

for i in range(10):
  loss = F.cross_entropy(inputs, labels, weight = weights)
print(f"functional = {loss}")

nn = 1.5734559297561646
functional = 1.5734559297561646


만약 위의 상황을 F.cross_entropy로 짠다면, weight를 매번 상황마다 넣어줘야 하고, nn.CrossEntropyLoss의 경우, 인스턴스화 이후 다시 대입할 필요는 없다

> 즉, 어트리뷰트를 이용한 추가적인 관리가 가능하다



### 2. 신경망을 정의하고 초기화하기

PyTorch를 사용해 모델을 생성할 때, 계산 그래프(즉, 신경망)에 데이터를 지나가게 하는 ``forward`` 함수를 정의해야 합니다.
이것은 feed-forward 알고리즘을 나타냅니다.

``forward`` 함수에서 어떠한 Tensor 연산자도 사용 가능합니다.




In [8]:
class Net(nn.Module):
    def __init__(self):
      super().__init__()
      #super(Net, self).__init__() #super 안에 적는 건 옛날 방식임
      # nn.Conv2d의 input: 입력 채널 수, 출력 채널 수, 컨볼루션 커널 크기, 패딩
      self.layer1 = nn.Sequential(nn.Conv2d(1,32,3,1),
                                  nn.BatchNorm2d(32),
                                  nn.Dropout2d(0.25))

      self.layer2 = nn.Sequential(nn.Conv2d(32,64,3,1),
                                  nn.BatchNorm2d(64),
                                  nn.Dropout(0.25))

      self.dropout1 = nn.Dropout2d(0.25)
      self.dropout2 = nn.Dropout2d(0.5)

      self.fc1 = nn.Linear(9216, 128)
      self.fc2 = nn.Linear(128, 10)

    # x는 데이터를 나타냅니다.
    def forward(self, x):
      x = self.layer1(x)
      x = F.relu(x)            # x를 ReLU 활성함수(rectified-linear activation function)에 대입합니다.

      x = self.layer2(x)
      x = F.relu(x)

      x = F.max_pool2d(x, 2)   # x에 대해서 max pooling을 실행합니다.
      x = self.dropout1(x)     # 데이터가 dropout1을 지나갑니다.
      x = torch.flatten(x, 1)  # start_dim=1으로 x를 압축합니다.

      x = self.fc1(x)          # 데이터가 fc1을 지나갑니다.
      x = F.relu(x)
      x = self.dropout2(x)
      x = self.fc2(x)

      output = F.log_softmax(x, dim=1)   # x에 softmax를 적용합니다.
      return output

Q. nn.Sequential vs nn.ModuleList

Q. torch.ravel, torch.flatten, torch.view(-1) 차이?

flatten = copy

ravel = 동일한 메모리 참조



In [9]:
a = torch.tensor([[[1, 2],
                   [3, 4]],
                  [[5, 6],
                   [7, 8]]])
a_ravel = torch.ravel(a)
a_flatten =torch.flatten(a)

print(f"a_ravel = {a_ravel}")
print(f"a_flatten = {a_flatten}")



a_ravel = tensor([1, 2, 3, 4, 5, 6, 7, 8])
a_flatten = tensor([1, 2, 3, 4, 5, 6, 7, 8])


### 2.1. 모델 구조 살펴보기

In [61]:
model = Net()
print(model)

Net(
  (layer1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Dropout2d(p=0.25, inplace=False)
  )
  (layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Dropout(p=0.25, inplace=False)
  )
  (dropout1): Dropout2d(p=0.25, inplace=False)
  (dropout2): Dropout2d(p=0.5, inplace=False)
  (fc1): Linear(in_features=9216, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


nn.modules를 상속받은 네트워크 클래스를 인스턴스화 한다.
__new__
__init__
__call__

**torchsummary를 이용한 모델 시각화**
> 가장 기본적인 라이브러리



> parameter 개수, 네트워크 구조 등을 확인 가능








In [2]:
uv pip install torchsummary

Note: you may need to restart the kernel to use updated packages.


c:\code\.venv\Scripts\python.exe: No module named uv


In [11]:
import torchsummary

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

torchsummary.summary(model, input_size=(1, 28, 28))

C:\code\.venv\Lib\site-packages\torch\nn\modules\dropout.py:176: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  return F.dropout2d(input, self.p, self.training, self.inplace)


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 26, 26]             320
       BatchNorm2d-2           [-1, 32, 26, 26]              64
         Dropout2d-3           [-1, 32, 26, 26]               0
            Conv2d-4           [-1, 64, 24, 24]          18,496
       BatchNorm2d-5           [-1, 64, 24, 24]             128
           Dropout-6           [-1, 64, 24, 24]               0
         Dropout2d-7           [-1, 64, 12, 12]               0
            Linear-8                  [-1, 128]       1,179,776
         Dropout2d-9                  [-1, 128]               0
           Linear-10                   [-1, 10]           1,290
Total params: 1,200,074
Trainable params: 1,200,074
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 1.41
Params size (MB): 4.58
Estimat

**torchinfo를 이용한 모델 시각화**
> nn.Sequential에 의해서 묶여있는 모듈들을 직관적으로 표현 가능







In [12]:
#%pip install torchinfo

from torchinfo import summary
batch_size = 16
summary(model, input_size=(batch_size, 1, 28, 28)) #input_size = (batch_size, channel, Height, Width)

Layer (type:depth-idx)                   Output Shape              Param #
Net                                      [16, 10]                  --
├─Sequential: 1-1                        [16, 32, 26, 26]          --
│    └─Conv2d: 2-1                       [16, 32, 26, 26]          320
│    └─BatchNorm2d: 2-2                  [16, 32, 26, 26]          64
│    └─Dropout2d: 2-3                    [16, 32, 26, 26]          --
├─Sequential: 1-2                        [16, 64, 24, 24]          --
│    └─Conv2d: 2-4                       [16, 64, 24, 24]          18,496
│    └─BatchNorm2d: 2-5                  [16, 64, 24, 24]          128
│    └─Dropout: 2-6                      [16, 64, 24, 24]          --
├─Dropout2d: 1-3                         [16, 64, 12, 12]          --
├─Linear: 1-4                            [16, 128]                 1,179,776
├─Dropout2d: 1-5                         [16, 128]                 --
├─Linear: 1-6                            [16, 10]                  1,290

**torchsummaryX를 이용한 모델 시각화**


> parameter 개수 뿐만아니라, Mult-Adds를 통해 덧셈과 곱셈을 총 몇 번 계산했는지도 확인 **가능**

> layer별 parameter와 Mult-Adds를 그래프로 시각화 해 주어서 어떤 계층이 연산량이 큰 지 확인 가능




In [16]:
# %pip install torchsummaryX

from torchsummaryX import summary

summary(model, torch.zeros(1, 1, 28, 28).to(device))


Note: you may need to restart the kernel to use updated packages.


C:\code\.venv\Lib\site-packages\torch\nn\modules\dropout.py:176: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  return F.dropout2d(input, self.p, self.training, self.inplace)


TypeError: unsupported operand type(s) for +: 'collections.OrderedDict' and 'collections.OrderedDict'

**<Pre-Trained 되어있는 다양한 모델들>**

In [18]:
import torchvision.models as models

# torchvision.models에 저장되어있는 모든 모델 리스트 가져오기 (너무 많이 출력되서 주석처리 해 두었습니다.)
"""
for model in models.list_models():
  print(model)
"""

# resnet 목록 다 가져오기 ( index 11 이전은 모두 모델이 아닌 weight 등의 다른게 저장되어 있었습니다. )
for resnet in models.resnet.__all__[11:]:
  print(resnet)

ModuleNotFoundError: No module named 'torchvision'

In [2]:
model = models.resnext101_64x4d(pretrained=True) # 위에서 출력한 resnet 리스트 중에서 보고싶은 모델을 가져옵니다

from torchsummary import summary
# from torchinfo import summary
# from torchsummaryX import summary

#아래에서 원하는 라이브러리의 주석만 해제하고 나머지는 주석처리하여 보고싶은 모델을 각자의 라이브러리로 시각화 해 봅시다.

"""
with torchsummary.summary:                                  # torchsummary로 출력
  summary(model, input_size=(1,28,28))
"""
summary(model.to('cuda'), input_size=(3,28,28))

"""
with torchinfo.summary:                                     #torchinfo로 출력
  summary(model, input_size=(batch_size, 1, 28, 28))
"""
# torchinfo.summary(model, input_size=(batch_size, 1, 28, 28))
"""
with torchsummaryX.summary:                                 #torchsummaryX로 출력
  summary(model, input_size=(batch_size, 1, 28, 28))
"""

NameError: name 'models' is not defined

In [21]:
print(model)

Net(
  (layer1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Dropout2d(p=0.25, inplace=False)
  )
  (layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Dropout(p=0.25, inplace=False)
  )
  (dropout1): Dropout2d(p=0.25, inplace=False)
  (dropout2): Dropout2d(p=0.5, inplace=False)
  (fc1): Linear(in_features=9216, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


### 3. 데이터를 모델에 적용해 출력 확인하기

원하는 출력값을 받을 수 있는 지 확인하기 위해, 무작위의 데이터를 모델에 통과시켜 시험해봅시다.




In [59]:
class Net(nn.Module):
    def __init__(self):
      super().__init__()
      #super(Net, self).__init__() #super 안에 적는 건 옛날 방식임
      # nn.Conv2d의 input: 입력 채널 수, 출력 채널 수, 컨볼루션 커널 크기, 패딩
      self.layer1 = nn.Sequential(nn.Conv2d(1,32,3,1),
                                  nn.BatchNorm2d(32),
                                  nn.Dropout2d(0.25))

      self.layer2 = nn.Sequential(nn.Conv2d(32,64,3,1),
                                  nn.BatchNorm2d(64),
                                  nn.Dropout(0.25))

      self.dropout1 = nn.Dropout2d(0.25)
      self.dropout2 = nn.Dropout2d(0.5)

      self.fc1 = nn.Linear(9216, 128)
      self.fc2 = nn.Linear(128, 10)

    # x는 데이터를 나타냅니다.
    def forward(self, x):
      x = self.layer1(x)
      x = F.relu(x)            # x를 ReLU 활성함수(rectified-linear activation function)에 대입합니다.

      x = self.layer2(x)
      x = F.relu(x)

      x = F.max_pool2d(x, 2)   # x에 대해서 max pooling을 실행합니다.
      x = self.dropout1(x)     # 데이터가 dropout1을 지나갑니다.
      x = torch.flatten(x, 1)  # start_dim=1으로 x를 압축합니다.

      x = self.fc1(x)          # 데이터가 fc1을 지나갑니다. Bx9216 ->Bx128
      x = F.relu(x)            # relu 함수를 적용시킵니다. Bx128 -> Bx128
      x = self.dropout2(x)
      x = self.fc2(x)

      output = F.log_softmax(x, dim=1)   # x에 softmax를 적용합니다.
      return output

In [60]:
# 임의의 28x28 이미지로 맞춰줍니다.
random_data = torch.rand((8, 1, 28, 28)) #(batch_size, channel, Height, Width)

my_nn = Net()
result = my_nn(random_data)
print(result) # output_size = (1,10)

tensor([[-2.7600, -1.4839, -1.9202, -2.8461, -2.5924, -3.1352, -2.5438, -2.8887,
         -2.3874, -1.8270],
        [-2.3413, -2.3397, -3.1349, -2.0641, -1.5182, -2.8499, -2.8312, -2.3390,
         -1.9425, -2.7914],
        [-2.6232, -1.1967, -2.4357, -3.2240, -1.8059, -3.4686, -2.3832, -2.4725,
         -2.6871, -2.8522],
        [-2.6556, -1.9516, -1.9867, -3.1260, -2.3562, -2.5172, -2.1170, -1.8842,
         -2.6993, -2.3903],
        [-2.5712, -2.4547, -3.4763, -2.3643, -1.3213, -3.2047, -2.2254, -1.7288,
         -2.8124, -2.8168],
        [-1.4079, -2.5538, -2.5279, -3.1610, -3.3837, -1.5365, -2.9169, -2.1526,
         -2.4427, -3.0142],
        [-1.8802, -1.6682, -2.5580, -2.9696, -3.4660, -2.7934, -2.3315, -1.6621,
         -2.4327, -2.7658],
        [-2.3617, -2.0797, -2.1904, -2.4396, -2.4132, -1.9652, -2.7593, -2.1985,
         -2.6139, -2.2586]], grad_fn=<LogSoftmaxBackward0>)


## [2교시] Loss
1. Loss의 개념
2. Pytorch 에서 Loss의 계산 실습
3. Loss를 최소화 하는 방법과 Gradient

### 1. Loss의 개념
 - loss는 모델의 예측인 $ŷ$ 과 실제 정답인 y 사이의 차이를 나타내는 지표입니다.
 - 실제 정답인 y는 변하지 않고, ŷ를 변화시켜  y와 가깝게 만드는 과정, 다시말해 Loss를 줄이는 과정을 학습이라고 합니다.
 - Loss를 구하는 function인 loss function을 어떻게 정의하느냐에 따라서 모델의 성능이 달라질 수 있습니다.




---
###Loss Function의 대표적인 두 함수

> $MSE(y, y_{pred}) = (1/n) * Σ_{i=1}^{n}(y_i - {y_{pred}}_i)^2$


> $CE(y, y_{pred}) = -Σ_{i=1}^{n} y_i * log({y_{pred}}_i)$


> y : 실제 정답 (label) $y_{pred}$ : 모델의 예측 (predict)









In [ ]:
# label = [0, 1, 0] <- [개, 고양이, 바다표범]
# input_dog = "dog" #원래는 픽셀값
# y_pred = model(input_dog) #y_pred = [0.7, 0.2, 0.1] label = [0, 1, 0]
#CE = 0*log(0.7) + 1*log(0.2) + 0*log(0.1)

###2. Pytorch에서 Loss의 계산 실습


In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 가상의 예측과 실제 값을 생성합니다.
predicted_values = torch.FloatTensor([0.2, 0.7, 0.1])  # 모델의 예측
true_values = torch.FloatTensor([1.0, 0.0, 0.0])       # 실제 라벨(정답)

# MSE 손실 계산
mse_loss = nn.MSELoss()
mse_output = mse_loss(predicted_values, true_values)
print("MSE Loss:", mse_output.item())

# CE 손실 계산
ce_loss = nn.CrossEntropyLoss()
ce_output = ce_loss(torch.sigmoid(predicted_values), true_values)
print("CE Loss:", ce_output.item())


MSE Loss: 0.3799999952316284
CE Loss: 1.1317548751831055


### Cross-entropy loss에 관하여


criterion = nn.CrossEntropyLoss(score, target)


$l(S,t)=−log[\frac{e^{S_t}}{\sum_{1}^{c}e^{S_c}}]$


근데 이걸 배치로 돌리면?

score = [Batch size(N) x the number of Class]
target = [Batch size(N)]


criterion은 N개의 input(이미지)에 대한 평균 loss값 → Scalar!

"즉 N개의 입력은 각각의 loss값을 가지는데 nn.CrossEntropyLoss는 8개의 loss값들을 모두 더해서 N으로 나눠줌"

**"들어온 score에 대해서는 1개의 input에 대한 score가 들어오든, n개의 input에 대한 score가 들어오든 모두 평균"**

In [25]:
score = torch.tensor( [0.8982,
                       0.805,
                       0.6393,
                       0.9983,
                       0.5731,
                       0.0469,
                       0.556,
                       0.1476,
                       0.8404,
                       0.5544] )

target = torch.tensor( 1 )

criterion = nn.CrossEntropyLoss()
print("방법1:original ",criterion(score,target))

print("방법2:manual ", torch.log(torch.sum(torch.exp(score))) - score[ target.item() ])

print("방법3:functional ",F.cross_entropy(score,target))

방법1:original  tensor(2.1438)
방법2:manual  tensor(2.1438)
방법3:functional  tensor(2.1438)


In [26]:
# 배치일때,

score = torch.randn(64,10)
target = torch.randint(9, size=(64,))

acc_loss = 0

for i in range(len(score)):

    loss = torch.log( torch.sum( torch.exp(score[i]) ) ) - score[i, target[i].item()]
    acc_loss += loss

print(acc_loss/ len(score))
criterion = nn.CrossEntropyLoss()
print(criterion(score, target))


tensor(2.7105)
tensor(2.7105)



Q. binary classification vs multi-class classification

https://returnclass.tistory.com/35

###3. Loss를 최소화 하는 방법과 Gradient
![이미지](https://editor.analyticsvidhya.com/uploads/631731_P7z2BKhd0R-9uyn9ThDasA.png)

출처 : https://www.analyticsvidhya.com/blog/2020/10/how-does-the-gradient-descent-algorithm-work-in-machine-learning/

---

**Gradient 계산 방법**



> $ g = \frac{\partial L(y, y_{pred})}{\partial w}$ (L : loss function)

**Optimize : Loss의 최소화 (3교시에 자세히)**


> $w = w - a*\frac{\partial L(y, y_{pred})}{\partial w}$
($a$ : learning_rate)










### ``torch.autograd`` 를 사용한 자동 미분
참고 : https://tutorials.pytorch.kr/beginner/blitz/autograd_tutorial.html

신경망을 학습할 때 가장 자주 사용되는 알고리즘은 **역전파** 입니다. 이 알고리즘에서,
매개변수(모델 가중치)는 주어진 매개변수에 대한 손실 함수의 **변화도(gradient)** 에
따라 조정됩니다.

이러한 변화도를 계산하기 위해 PyTorch에는 ``torch.autograd`` 라고 불리는 자동 미분 엔진이
내장되어 있습니다. 이는 모든 계산 그래프에 대한 변화도의 자동 계산을 지원합니다.

입력 ``x``, 매개변수 ``w`` 와 ``b`` , 그리고 일부 손실 함수가 있는 가장 간단한 단일 계층
신경망을 가정하겠습니다. PyTorch에서는 다음과 같이 정의할 수 있습니다:


In [27]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

### requires_grad


 ``w`` 와 ``b`` 는 최적화를 해야 하는 **매개변수** 입니다. 따라서
이러한 변수들에 대한 손실 함수의 변화도를 계산할 수 있어야 합니다. 이를 위해서 해당 텐서에
``requires_grad`` 속성을 설정합니다.



<div class="alert alert-info"><h4>Note</h4><p>
requires_grad 의 값은 텐서를 생성할 때 설정하거나, 나중에
          x.requires_grad_(True) 메소드를 사용하여 나중에 설정할 수도 있습니다.</p></div>



### 변화도(Gradient) 계산하기

신경망에서 매개변수의 가중치를 최적화하려면 매개변수에 대한 손실함수의 도함수(derivative)를
계산해야 합니다. 즉, ``x`` 와 ``y`` 의 일부 고정값에서 $\frac{\partial loss}{\partial w}$ 와
$\frac{\partial loss}{\partial b}$ 가 필요합니다.
이러한 도함수를 계산하기 위해, ``loss.backward()`` 를 호출한 다음 ``w.grad`` 와
``b.grad`` 에서 값을 가져옵니다:




In [ ]:
loss.backward()
print(w.grad)
print(b.grad)

<div class="alert alert-info"><h4>Note</h4><p>
  
  - 연산 그래프의 잎(leaf) 노드들 중 ``requires_grad`` 속성이 ``True`` 로 설정된
    노드들의 ``grad`` 속성만 구할 수 있습니다. 그래프의 다른 모든 노드에서는 변화도가
    유효하지 않습니다.

  - 성능 상의 이유로, 주어진 그래프에서의 ``backward`` 를 사용한 변화도 계산은 한 번만
    수행할 수 있습니다. 만약 동일한 그래프에서 여러번의 ``backward`` 호출이 필요하면,
    ``backward`` 호출 시에 ``retrain_graph=True`` 를 전달해야 합니다.</p></div>




### 변화도 추적 멈추기

기본적으로, ``requires_grad=True`` 인 모든 텐서들은 연산 기록을 추적하고 변화도 계산을
지원합니다. 그러나 모델을 학습한 뒤 입력 데이터를 단순히 적용하기만 하는 경우와 같이 *순전파*
연산만 필요한 경우에는, 이러한 추적이나 지원이 필요 없을 수 있습니다.
연산 코드를 ``torch.no_grad()`` 블록으로 둘러싸서 연산 추적을 멈출 수 있습니다:




In [28]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

True
False


동일한 결과를 얻는 다른 방법은 텐서에 ``detach()`` 메소드를 사용하는 것입니다:




In [29]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

False


변화도 추적을 멈춰야 하는 이유들은 다음과 같습니다:
  - 신경망의 일부 매개변수를 **고정된 매개변수(frozen parameter)** 로 표시합니다.
  - 변화도를 추적하지 않는 텐서의 연산이 더 효율적이기 때문에, 순전파 단계만 수행할 때
    **연산 속도가 향상됩니다.**


Loss 관련 좋은 자료:
https://www.v7labs.com/blog/pytorch-loss-functions



## [3교시] Optimizer


![이미지](https://velog.velcdn.com/images/olxtar/post/3c076577-5925-4ce7-9e4a-320e9737e43c/2dKCQHh.gif)


> 출처 : https://velog.io/@freesky/Optimizer



---


###  *Stochastic Gradient Descent (SGD)*

> $w = w - a*\frac{\partial L(y, y_{pred})}{\partial w}$
($a$ : learning_rate)

> 의의 : 가장 기본적인 optimize 방법

> 한계 : local minimum에 수렴하게 되면 빠져나올 수 없다.













In [30]:
learning_rate = 1e-3
optimizer = torch.optim.SGD(model.parameters(),lr = learning_rate)

###local minimum

![이미지](https://blog.kakaocdn.net/dn/4K39d/btqCJmMMRsx/cz2mKqhbhXZMqmpGDlNEYk/img.png)

출처 : https://sungwookkang.com/1416

### *Momentum*

> $
  \begin{align*}
  m_t &= \beta \cdot m_{t-1} + (1 - \beta) \cdot \nabla J(w_t) \\
  w_{t+1} &= w_t - \alpha \cdot m_t
  \end{align*}
  $

> 의의 : 이전 step때 갖고있던 gradient로 관성을 만들어 local minimum을 빠져나옴








In [31]:
#momentum은 모듈이 따로 제작되지 않고 parameter로 구현되어있습니다.
learning_rate = 1e-3
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate, momentum = 0.1)

### *Adaptive gradient (AdaGrad)*

>$ \begin{align*}
  G_t &= G_{t-1} + (\nabla J(w_t))^2 \\
  w_{t+1} &= w_t - \frac{\alpha}{\sqrt{G_t + \epsilon}} \cdot \nabla J(w_t)
  \end{align*}$

> 의의 : 학습과정의 gradient를 제곱하여 누적 합을 learning rate의 분모에 넣었음

> - 누적 합이 작은 경우 : gradient가 작으므로 minimum 근처라고 판단, learning rate를 증가시켜 optimize를 더 빠르게 하거나, local minimum을 빠져나오게 됨

> - 누적 합이 큰 경우 : gradient가 크므로, 경사가 급하다고 판단, learning rate를 감소시킴으로써 수렴 속도를 조금 늦춰 정확도를 높여줌






In [32]:
learning_rate = 1e-3
optimizer = torch.optim.Adagrad(model.parameters(), lr = learning_rate)

### *Adaptive gradient + Momentum (Adam)*

>$\begin{align*}
  m_t &= \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot \nabla J(w_t) \\
  v_t &= \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot (\nabla J(w_t))^2 \\ \\
    \hat{m}_t &= \frac{m_t}{1 - \beta_1^t} \\
  \hat{v}_t &= \frac{v_t}{1 - \beta_2^t} \\ \\
  w_{t+1} &= w_t - \frac{\alpha}{\sqrt{\hat{v}_t + \epsilon}} \cdot \hat{m}_t \\
  \end{align*}$

> $\hat{m}_{t}$ : Momentum

> $\hat{v}_{t}$ : Adaptive gradient

> $\beta_1$ = 0.9, $\beta_2$ = 0.999 가 추천됨

> 의의 : 좋은 성능을 보였던 Optimizer들을 한데 모았음

In [33]:
learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate, betas = (0.9, 0.999)) #betas = (momentum, adagrad)

https://pytorch.org/docs/stable/optim.html
> 다양한 optimizer에 대한 설명을 볼 수 있는 Pytorch 공식 홈페이지

> Optimizer에 들어가는 parameter, optimize 규칙을 수식으로 확인 할 수 있다.

##[4교시] Weight Initialize
- 이거 우리가 직접 안 하고 He 계열이 지금 토치 기본으로 사용되는 중
![이미지](https://user-images.githubusercontent.com/45377884/91631652-2fc76c00-ea16-11ea-98f2-3dc11c0f5f43.png)

참고 : https://yngie-c.github.io/deep%20learning/2020/03/17/parameter_init/

### Zero Initialize
 *모든 가중치를 0으로 초기화*
 - 네트워크의 모든 weight가 동일하게 초기화되므로, 연산의 다양성이 줄어든다
 - input이 들어와도 한 레이어 안의 여러 노드가 똑같은 출력값을 내놓는다는 뜻
 - 여러 노드를 사용하는 의미가 없어지고, 결과적으로 한 개의 노드만 레이어에 넣는것과 같은 효과를 가짐

 > 비추 !

In [34]:
import torch

layer = torch.nn.Conv2d(1, 1, 2)

# Normal distribution
torch.nn.init.normal_(layer.weight)

Parameter containing:
tensor([[[[-1.2321,  0.1351],
          [ 0.4019,  0.5821]]]], requires_grad=True)

### Random Initialize
*모든 가중치를 평균과 표준편차를 이용한 확률분포에 따라 초기화*
- sigmoid를 지나자 0과 1 양 극단으로 치우쳐버려 학습이 안됨

![이미지](https://user-images.githubusercontent.com/45377884/91642462-852b6980-ea66-11ea-84cd-2cdbc3007db1.png)

- 위 양극화 문제를 해결하기 위해 편차를 굉장히 줄였더니 하나의 값으로 수렴하여 zero initialize와 같은 문제가 발생함

![이미지](https://user-images.githubusercontent.com/45377884/91642484-aab87300-ea66-11ea-94ba-f0236b3fe508.png)

### Xavier Initialize
>$\text{Var}(Y) = \text{Var}(XW)$

- x와 w는 독립이므로 서로 분리가능

>$\text{Var}(X_{i}W_{i}) = \text{Var}(X_{i}) \text{Var}(W_{i})$

>$\text{Var}(Y) = n\text{Var}(X) \text{Var}(W)$

- weight initializer는 input과 output의 분산이 같음을 목표로 함
- 이 때 W의 분산은?

>$\text{Var}(W) = 1/n $
- **이전 계층의 노드수에 반비례하게 현재 계층의 weight의 분산을 설정하여 초기화**

![이미지](https://user-images.githubusercontent.com/45377884/91678679-24dd1a80-eb81-11ea-9637-abcdbcd3788c.png)

In [35]:
import torch

layer = torch.nn.Conv2d(1, 1, 2)

# Xavier initialization
torch.nn.init.xavier_uniform_(layer.weight)


Parameter containing:
tensor([[[[-0.2708, -0.1326],
          [-0.2934,  0.2237]]]], requires_grad=True)

*사람이름 : Xavier Glorot*

**TensorFlow**

tf.keras.initializers.HeNormal()

**PyTorch**

torch.nn.init.xavier_normal_()

### He Initialize
위 Xavier Initializer에서 분산을 2배로 늘린 것

![이미지](https://user-images.githubusercontent.com/45377884/91679856-5efbeb80-eb84-11ea-870b-9fe512141ca5.png)

- relu는 0 이하의 값들을 모두 0으로 바꿔버리므로, 계층을 통과할수록 0에 많은 값이 포함되게 된다. -> 분산을 2배로 늘리자!

![이미지](https://user-images.githubusercontent.com/45377884/91680106-34f6f900-eb85-11ea-8abe-7bd830483f21.png)

- relu의 특성상 0에 몰려있지만 해당 사실은 두 그래프에 굳이 나타내지 않은 듯 하다. 대신 0 이외의 값들에 대해서는 균일하게 initialize 된 것을 볼 수 있다.

In [ ]:
import torch

layer = torch.nn.Conv2d(1, 1, 2)

# Kaiming initialization
torch.nn.init.kaiming_uniform_(layer.weight)

###네트워크에서 initializer 바꾸기

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128) #784
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

        torch.nn.init.kaiming_uniform_(self.fc1.weight)

    def forward(self, x):
        x = x.view(-1, 28 * 28)  # Flatten the input ->784
        x = torch.relu(self.fc1(x)) #784 -> 128 -> relu ::pos
        x = torch.relu(self.fc2(x)) #128 -> 64 -> relu ::pos
        x = self.fc3(x)             #64 -> 10
        return x
net = Net()

##[5교시] Multi Layer Perceptron for MNIST

### 라이브러리 불러오기

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

### Device를 선언
> cpu를 사용할지 gpu를 사용할지

In [ ]:
# GPU를 사용할 수 있는 경우 사용하고, 그렇지 않으면 CPU를 사용합니다.
if torch.cuda.is_available():
    device_name = "cuda:0"
else:
    device_name =  "cpu"
device = torch.device(device_name)

print(device)

###데이터 전처리 및 로드

In [ ]:
# 데이터 전처리 및 로드
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))])

batch_size = 64

"""
batch: dataloader가 한번에 들고오는 데이터 개수
shuffle: 섞는것
epoch: 모든 데이터셋을 한번 다 보는 경우 = 1epoch


전체 1000장, batch_size=10장, 데이터로더 100번 움직여야 :: mini-mini-batch (장려X)
전체 1000장, batch_size=100장, 데이터로더 10번 움직여야 :: mini-batch (장려)
전체 1000장, batch_size=1000장, 데이터로더 1번 움직여야 :: full-batch (장려)
"""
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


MNIST
![](https://upload.wikimedia.org/wikipedia/commons/2/27/MnistExamples.png)

Q.
환자 100명의 데이터, 사람마다 데이터 개수는 달라요
총 10000장
> a:9900장 100장
train vs test: 8000장(shuffle) vs 2000장 (a:2000장)
train vs test: 80명 vs 20명
train vs test: 95명 vs 5명

>> 물리적으로 격리된 신규 데이터에서도 충분한 성능을 내는! 모델 개발이 중요하다

### 네트워크 정의

![](https://static.javatpoint.com/tutorial/tensorflow/images/multi-layer-perceptron-in-tensorflow.png)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)  # Flatten the input
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

###네트워크를 선언한 Device로 옮기기

In [ ]:
net = Net()
net.to(device)

### Loss 및 Optimizer 선언

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

### Train Loop

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(device) # batch_size x C1 x H28 x W28   !=   batch_size x h x w x c  (tensorflow)
        labels = labels.to(device) # batch_size x 1
        #print(labels.size())
        optimizer.zero_grad() #parameter가 이전 step에서 사용한 grad를 0으로 만들어줌 *momentum의 경우 별도로 저장해서 안날아감
        outputs = net(inputs) #outputs (batch_size x 10)
        loss = loss_fn(outputs, labels)
        #print(loss)
        loss.backward()      # gradient 계산
        optimizer.step()     # 계산된 gradient로 wight를 update
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader)}")

print("Finished Training")

### Test Loop

In [ ]:
correct = 0
total = 0

net.eval()  #net.train()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = net(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on the test images: {accuracy}%")

## 코드 리뷰


In [ ]:
%matplotlib inline

In [ ]:
# -*- coding: utf-8 -*-
import random
import torch
import math
import matplotlib.pyplot as plt

class DynamicNet(torch.nn.Module):
    def __init__(self):
        """
        In the constructor we instantiate five parameters and assign them as members.
        """
        super().__init__()
        self.a = torch.nn.Parameter(torch.randn(()))
        self.b = torch.nn.Parameter(torch.randn(()))
        self.c = torch.nn.Parameter(torch.randn(()))
        self.d = torch.nn.Parameter(torch.randn(()))
        self.e = torch.nn.Parameter(torch.randn(()))

    def forward(self, x):
        """
        For the forward pass of the model, we randomly choose either 4, 5
        and reuse the e parameter to compute the contribution of these orders.

        Since each forward pass builds a dynamic computation graph, we can use normal
        Python control-flow operators like loops or conditional statements when
        defining the forward pass of the model.

        Here we also see that it is perfectly safe to reuse the same parameter many
        times when defining a computational graph.
        """
        y = self.a + self.b * x + self.c * x ** 2 + self.d * x ** 3
        for exp in range(4, random.randint(4, 6)): #4,6 포함 0 or 4,5 or 4,5,6
            y = y + self.e * x ** exp
        return y

    def string(self):
        """
        Just like any class in Python, you can also define custom method on PyTorch modules
        """
        return f'y = {self.a.item()} + {self.b.item()} x + {self.c.item()} x^2 + {self.d.item()} x^3 + {self.e.item()} x^4 ? + {self.e.item()} x^5 ?'


# Create Tensors to hold input and outputs.
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

# Construct our model by instantiating the class defined above
model = DynamicNet()

# Construct our loss function and an Optimizer. Training this strange model with
# vanilla stochastic gradient descent is tough, so we use momentum
criterion = torch.nn.MSELoss(reduction='sum')
optimizer = torch.optim.SGD(model.parameters(), lr=1e-8, momentum=0.9) #
for t in range(10000):
    # Forward pass: Compute predicted y by passing x to the model
    y_pred = model(x)

    # Compute and print loss
    loss = criterion(y_pred, y)
    if t % 2000 == 1999:
        print(f"Epoch = {t}, loss = {loss.item()}")

    # Zero gradients, perform a backward pass, and update the weights.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f'Result: {model.string()}')

y_pred = model(x).detach().numpy()
x = x.detach().numpy()

plt.plot(x, y, "r")
plt.plot(x, y_pred, "b")
plt.show()